In [ ]:
import pandas as pd
from google.cloud import bigquery
import unicodedata
import re
from thefuzz import process, fuzz
client = bigquery.Client()

In [ ]:
id_query = """
SELECT code, first_name, second_name, understat, `16-17`, `17-18`, `18-19`, `19-20`, `20-21`, `21-22`, `22-23`, `23-24`, `24-25`, `25-26`
FROM `fpl-optima.fpl_bronze.id_map`
"""
fpl_query = """
SELECT element, name, position, team, clean_sheets, assists, goals_scored, goals_conceded, defensive_contribution,expected_goals, expected_assists, ict_index, minutes, value/10 as price, opponent_team, total_points, bonus,xp, gw, season, kickoff_time
FROM `fpl-optima.fpl_bronze.fpl`
"""
id_query_job = client.query(id_query)
id_df = id_query_job.to_dataframe()
fpl_query_job = client.query(fpl_query)
fpl_df = fpl_query_job.to_dataframe()

In [ ]:
fpl_df = fpl_df[fpl_df['position'] != "AM"] # Removing managers from the dataset

In [ ]:
# Dealing wwith name inconsistencies (Ingested incorrectly from the API, and some players have accents in their names)
def clean_name(name):
    if not name:return ""
    try:
        name = name.encode('latin-1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('ascii')
    name = re.sub(r'\d+$', '', name)
    return name.lower().replace("_", " ").strip()

fpl_df['clean_name'] = fpl_df['name'].apply(clean_name)

id_df['full_name'] = id_df['first_name'] + " " + id_df['second_name']
id_df['full_name'] = id_df['full_name'].apply(clean_name)

fpl_manual_fixes = {
    'edward nketiah': 'eddie nketiah',
    'damian emiliano martinez': 'emiliano martinez',
    'bamidele alli': 'dele alli',
    'maximillian aarons': 'max aarons',
    'matthew cash': 'matty cash',
    'treymaurice nyoni': 'trey nyoni',
    'benjamin chilwell': 'ben chilwell',
    'frederick woodman': 'freddie woodman',
    'ben white': 'benjamin white',
    'olayinka fredrick oladotun ladapo': 'freddie ladapo',
    'pelenda joshua dasilva': 'josh dasilva',
    'matthew james': 'matty james',
    'ajibola alese': 'aji alese',
    'bonatini lohner maia bonatini': 'leo bonatini',
    'cuco martina': 'rhu-endly martina',
    'diogo jota': 'diogo teixeira da silva',
    'max kilman': 'maximilian kilman',
    'borja baston': 'borja gonzalez tomas',
    'fabio silva': 'fabio soares silva',
    'kaine hayden': 'kaine kesler-hayden', #
    'solomon march': 'solly march',
    'heung-min son': 'son heung-min',
    'yegor yarmolyuk': 'yehor yarmoliuk',
    'yegor yarmoliuk': 'yehor yarmoliuk',
    'joao pedro ferreira silva': 'joao pedro ferreira da silva',
    'alexandre nascimento costa silva': 'xande nascimento da costa silva',
    'philippe coutinho': 'philippe coutinho correia',
    'joseph gomez': 'joe gomez',
}


fpl_df['clean_name'] = fpl_df['clean_name'].replace(fpl_manual_fixes)

In [ ]:
## More manual fixes like name changes, and removing duplicates

id_df.loc[id_df['full_name'] == "yehor yarmoliuk", '22-23'] = 650.0
id_df = id_df[~id_df['full_name'].str.contains("yegor yarmolyuk", case=False, na=False)]

for index, row in fpl_df.iterrows():
    if row['clean_name'] == "joshua king" and row['team'] == "Fulham":
        fpl_df.loc[index, 'clean_name'] = "josh King"

for index, row in fpl_df.iterrows():
    if row['clean_name'] == "alvaro fernandez" and row['team'] == "Man Utd":
        fpl_df.loc[index, 'clean_name'] = "alvaro fernandez carreras"

id_df.loc[id_df['full_name'] == "kaine kesler-hayden", '22-23'] = 537.0
id_df.loc[id_df['full_name'] == "kaine kesler-hayden", '19-20'] = 659.0
id_df = id_df[~id_df['code'].isin([490098, 537043])]

In [ ]:
##### Check for duplicate full names in id_df
# Droped the duplicates that are no more active, and have no data for the seasons we care about. 
duplicates = id_df[id_df.duplicated(subset=['full_name'], keep=False)].sort_values('full_name')
print(f"Total duplicate full names: {duplicates['full_name'].nunique()}")
print(f"Total duplicate rows: {len(duplicates)}")
print("\nDuplicate entries:")
display(duplicates)

for index, row in id_df.iterrows():
    if row['code'] in ([41792,152898,75826]):
        id_df.drop(index, inplace=True)

duplicates = id_df[id_df.duplicated(subset=['full_name'], keep=False)].sort_values('full_name')
print(f"Total duplicate full names: {duplicates['full_name'].nunique()}")
print(f"Total duplicate rows: {len(duplicates)}")
print("\nDuplicate entries:")
display(duplicates)

In [ ]:
cond1 = (fpl_df['clean_name'] == "aaron ramsey") & (fpl_df['element'].isin([16, 18, 14]))
cond2 = (fpl_df['clean_name'] == "ben davies") & (fpl_df['element'].isin([653, 248, 499]))
cond3 = (fpl_df['clean_name'] == "danny ward") & (fpl_df['element'].isin([105]))

fpl_df = fpl_df[~(cond1 | cond2 | cond3)].reset_index(drop=True)

In [ ]:
# THE MERGE>>> Wohoo!!
id_season_cols = [col for col in id_df.columns if '-' in col]
merged_df = fpl_df.merge(
            id_df[['code', 'full_name', 'understat'] + id_season_cols ],
            left_on='clean_name', 
            right_on='full_name', 
            how='left'
            )

merged_df['season_short'] = merged_df['season'].str[-5:]

In [ ]:
# Matching names using fuzzy match for the unmateched ones
unmatched_mask = merged_df['full_name'].isna()
unmatched_names = merged_df.loc[unmatched_mask, 'clean_name'].unique()
choices = id_df['full_name'].tolist()

def get_best_match_v3(name, choices):
    if not name: return None

    match, score = process.extractOne(name, choices, scorer=fuzz.token_set_ratio)
    
    if score >= 95:
        return match

    match_v2, score_v2 = process.extractOne(name, choices, scorer=fuzz.token_sort_ratio)
    
    return match_v2 if score_v2 >= 80 else None

#def get_best_match_hybrid(name, choices):
    if not name: 
        return None
    
    match, score = process.extractOne(name, choices, scorer=fuzz.token_set_ratio)
    if score >= 90:
        return match

    p_match, p_score = process.extractOne(name, choices, scorer=fuzz.partial_token_set_ratio)
    
    if p_score >= 98:
        return p_match
        
    return None

# def get_best_match(name):
    if not name: return None
    match, score = process.extractOne(name, choices, scorer=fuzz.token_set_ratio)
    return match if score >= 85 else None

name_map = {name: get_best_match_v3(name, choices) for name in unmatched_names if name}

merged_df.loc[unmatched_mask, 'fuzzy_name'] = merged_df.loc[unmatched_mask, 'clean_name'].map(name_map)


#id_df_unique = id_df.drop_duplicates(subset=['full_name'], keep='first') 
#id_lookup = id_df_unique.set_index('full_name')
id_lookup = id_df.set_index('full_name').to_dict()

for col in ['code', 'understat'] + id_season_cols:
    merged_df[col] = merged_df[col].fillna(merged_df['fuzzy_name'].map(id_lookup[col]))

In [ ]:
# Look at the names that didn't find a match
missing_names = merged_df[merged_df['code'].isna()]['clean_name'].unique()

print(f"Total unique names in FPL: {fpl_df['clean_name'].nunique()}")
print(f"Names failed to match: {len(missing_names)}")

In [ ]:
# Check if the element of each player matches the season column in Chris Musson's id map table table w.r.t the season of the FPL data. (Hope, it made sense)
def season_id(row):
    season_col = row['season_short']
    if season_col in row and pd.notnull(row[season_col]):
            return row['element'] == row[season_col]
    return False

merged_df['season_match'] = merged_df.apply(season_id, axis=1)

In [ ]:
#merged_df.groupby('code')['understat'].nunique() #Checking for wrong code x understat matches, which shouldn't happen.


In [ ]:
merged_df[merged_df['season_match'] == False]

In [ ]:
print(fpl_df.shape)
print(merged_df.shape)

In [ ]:
# Appending the columns we need
fpl_df['code'] = merged_df.loc[merged_df['season_match'], 'code']
fpl_df['understat'] = merged_df.loc[merged_df['season_match'], 'understat']

In [ ]:
# Obviously, only Ibra has no team/position data, cause he's Ibra
fpl_df.loc[fpl_df['clean_name'] == "zlatan ibrahimovic", ['position', 'team']] = ["FWD", "Man Utd"]

In [ ]:
old_count = fpl_df['clean_name'].nunique()

fpl_df = fpl_df.sort_values(['code', 'kickoff_time'], ascending=[True, False])

latest_names = fpl_df.groupby('code')['clean_name'].first()

fpl_df['clean_name'] = fpl_df['code'].map(latest_names)

new_count = fpl_df['clean_name'].nunique()

print(f"Unique names before: {old_count}")
print(f"Unique names after:  {new_count}")
print(f"Names 'consolidated': {old_count - new_count}")

In [ ]:
fpl_df[fpl_df['team'].isna() | fpl_df['position'].isna()]['clean_name'].unique()

In [ ]:
fpl_df['season_start_year'] = fpl_df['season'].str.split('-').str[0].astype(int)

In [ ]:
# Arranging tthe data (OCD issues)

fpl_df['kickoff_time'] = pd.to_datetime(fpl_df['kickoff_time'])


fpl_df = fpl_df.sort_values(by=['season_start_year', 'gw', 'kickoff_time'], ascending=[True, True, True])

In [ ]:
fpl_df.to_csv("c:\\Programing\\Projects\\Fantasy Premier League\\data\\fpl_silver.csv", index=False)

In [ ]:
## Pushing it to BigQuery after fixing null position and team values for some players
fpl_df.columns = [c.replace(' ', '_').lower() for c in fpl_df.columns]

TABLE_ID = "fpl-optima.fpl_silver.fpl_mapped"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
    autodetect=True  
)

print(f"Uploading {len(fpl_df)} rows to {TABLE_ID}...")

job = client.load_table_from_dataframe(fpl_df, TABLE_ID, job_config=job_config)
job.result() 

print("Silver Table Created Successfully!")